# Exercise 2: Collecting errors

**Level: introductory worked exercise.** The “Your turn” task is an extension.

*Generation and Dialogue · version 0.1.0*

> Ask a model 25 culturally grounded questions each in two Turkic languages (holidays, food, place names, proverbs). Measure language, script and analyser coverage automatically, then classify the errors in the answers with the seven types of the chapter.

**Setup.** `pip install llama-cpp-python fasttext-wheel requests "turkicnlp[hfst]==0.5.0"` (the compiled `hfst` package needs Python 3.11 or 3.12). The generator is **Qwen2.5-1.5B-Instruct** in 8-bit GGUF form, run on the CPU with llama.cpp: set `QWEN_GGUF` to the file. The book's run converted the Hugging Face checkpoint at revision `989aa79` with llama.cpp's `convert_hf_to_gguf.py --outtype q8_0` (SHA-256 recorded in the outputs); Qwen's own `Qwen2.5-1.5B-Instruct-GGUF` q8_0 file is an equivalent starting point. Language identification uses GlotLID v3 (`GLOTLID_PATH`, otherwise downloaded from the Hub), morphology the Apertium analysers that come with TurkicNLP. Decoding is greedy (temperature 0) and every model reply is cached in `BOOK_DATA_DIR` (default `~/.cache/turkic-nlp-book`), so the outputs can be inspected and a rerun is exact. **One small model is used throughout.** Its errors illustrate the measurement methods; they are not a ranking of models or a statement about larger ones.

The questions were written for this exercise in Kazakh (Cyrillic) and Uzbek (Latin). Each answer is split into sentences, labelled by GlotLID over its full label set, measured for the share of letters in the requested script, and analysed with Apertium.

In [ ]:
import collections, hashlib, json, math, os, random, re, statistics, unicodedata
from pathlib import Path
import requests

CACHE = Path(os.environ.get("BOOK_DATA_DIR", "~/.cache/turkic-nlp-book")).expanduser()
THREADS = int(os.environ.get("THREADS", os.cpu_count() or 1))
QWEN_GGUF = os.environ.get("QWEN_GGUF", str(CACHE / "models/qwen2.5-1.5b-instruct-q8_0.gguf"))
SIB = "https://raw.githubusercontent.com/dadelani/sib-200/efad32e0fc601cfbfdf5ca80672f29697f6f401f/data/annotated/"
KAZNERD = "https://raw.githubusercontent.com/IS2AI/KazNERD/bd4333d0f5952b9fafb2ef2ac2fefa0ad3c0333f/KazNERD/IOB2_test.txt"
FETCHED = {}

def fetch(url):
    path = CACHE / hashlib.sha1(url.encode()).hexdigest()[:12] / url.rsplit("/", 1)[-1]
    if not path.exists():
        r = requests.get(url, timeout=300); r.raise_for_status()
        path.parent.mkdir(parents=True, exist_ok=True); path.write_bytes(r.content)
    FETCHED[url] = hashlib.sha256(path.read_bytes()).hexdigest()
    return path

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

# ---- the generator, with every reply cached ---------------------------------------------------
_LLM = {}
REPLIES = CACHE / "ch14-replies.jsonl"
_CACHED = {}
if REPLIES.exists():
    for line in REPLIES.read_text(encoding="utf-8").splitlines():
        r = json.loads(line); _CACHED[r["key"]] = r["reply"]

def chat(messages, max_tokens=200):
    """Greedy reply of Qwen2.5-1.5B-Instruct (llama.cpp, q8_0) to a list of chat messages."""
    key = hashlib.sha1(json.dumps([messages, max_tokens], ensure_ascii=False).encode()).hexdigest()
    if key not in _CACHED:
        if "m" not in _LLM:
            from llama_cpp import Llama
            _LLM["m"] = Llama(model_path=QWEN_GGUF, n_ctx=4096, n_threads=THREADS, seed=0, verbose=False)
        out = _LLM["m"].create_chat_completion(messages=messages, temperature=0.0, max_tokens=max_tokens)
        _CACHED[key] = out["choices"][0]["message"]["content"].strip()
        REPLIES.parent.mkdir(parents=True, exist_ok=True)
        with REPLIES.open("a", encoding="utf-8") as f:
            f.write(json.dumps({"key": key, "messages": messages, "reply": _CACHED[key]}, ensure_ascii=False) + "\n")
    return _CACHED[key]

# ---- automatic fidelity signals ---------------------------------------------------------------
_LID = {}
def lid(text):
    """Unrestricted GlotLID v3 label and probability of one line of text."""
    if "m" not in _LID:
        import fasttext
        path = os.environ.get("GLOTLID_PATH")
        if not path:
            from huggingface_hub import hf_hub_download
            path = hf_hub_download("cis-lmu/glotlid", "model_v3.bin")
        _LID["m"] = fasttext.load_model(path)
    labels, probs = _LID["m"].predict([text.replace("\n", " ")], k=1)
    return labels[0][0].replace("__label__", ""), float(probs[0][0])

SENT = re.compile(r"(?<=[.!?…])\s+|\n+")
def sentences(text):
    return [s.strip() for s in SENT.split(text) if len(re.findall(r"[^\W\d_]", s)) >= 3]

def script_shares(text):
    letters = [c for c in text if c.isalpha()]
    count = collections.Counter("Cyrl" if "CYRILLIC" in unicodedata.name(c, "") else
                                "Latn" if "LATIN" in unicodedata.name(c, "") else "other" for c in letters)
    return {k: v / len(letters) for k, v in count.items()} if letters else {}

WORD = re.compile(r"[^\W\d_]+")
_FST = {}
def lookup(lang):
    """Raw Apertium analyser of TurkicNLP for kaz (Cyrillic) or uzb (Latin): word -> [(reading, weight)]."""
    if lang not in _FST:
        import hfst, turkicnlp
        from turkicnlp.resources.registry import ModelRegistry
        script = {"kaz": "Cyrl", "uzb": "Latn"}[lang]
        turkicnlp.download(lang, processors=["morph"])
        path = ModelRegistry.get_model_path(lang, "morph", "apertium", script=script) / f"{lang}.automorf.hfst"
        fst = hfst.HfstInputStream(str(path)).read()
        _FST[lang] = (lambda w: fst.lookup(w) or fst.lookup(w.lower())), sha256(path)
    return _FST[lang][0]

def accepted(word, lang):
    return any(r and not r.startswith("*") and math.isfinite(w) for r, w in lookup(lang)(word))

def coverage(texts, lang):
    words = [w for t in texts for w in WORD.findall(t)]
    ok = sum(accepted(w, lang) for w in words)
    return ok, len(words)

print("generator:", Path(QWEN_GGUF).name, "| sha256", sha256(QWEN_GGUF)[:16] if Path(QWEN_GGUF).exists() else "missing")

In [ ]:
QUESTIONS = {
 "kaz": ["Наурыз мейрамы қашан және қалай тойланады?", "Бешбармақ қалай дайындалады?", "Домбыра деген не?",
         "Абай Құнанбайұлы кім болған?", "Қазақстанның астанасы қай қала?", "Қымыз неден жасалады?",
         "Қазақтың киіз үйі қалай құрылады?", "Тәуелсіздік күні қашан аталып өтеді?", "Алматы қаласы немен танымал?",
         "«Еңбек етсең ерінбей, тояды қарның тіленбей» мақалының мағынасы қандай?", "Бауырсақ деген не?",
         "Қазақстанның ұлттық валютасы қалай аталады?", "Көкпар ойыны қалай ойналады?", "Шоқан Уәлиханов кім?",
         "Қазақ тілінде үлкен кісіге қалай сыпайы жүгінеді?", "Қазақстандағы ең биік тау қайсы?",
         "Балқаш көлінің ерекшелігі неде?", "Айтыс деген не?", "Құрманғазы кім болған?", "Қазақстанда қандай ұлттық спорт түрлері бар?",
         "Тұсаукесер дәстүрі деген не?", "Қазақ хандығы қашан құрылған?", "Шымкент қаласы туралы айтып беріңізші.",
         "Қазақтың ұлттық тағамдарын атаңыз.", "Ораза айт мерекесі қалай тойланады?"],
 "uzb": ["Navro‘z bayrami qachon va qanday nishonlanadi?", "O‘zbek oshi qanday tayyorlanadi?", "Do‘ppi nima?",
         "Alisher Navoiy kim bo‘lgan?", "O‘zbekistonning poytaxti qaysi shahar?", "Somsa qanday pishiriladi?",
         "Samarqand nimasi bilan mashhur?", "Mustaqillik kuni qachon nishonlanadi?", "Buxoro shahri haqida gapirib bering.",
         "“Bilmaganni so‘rab o‘rgangan olim” maqolining ma’nosi nima?", "Non O‘zbekistonda qanday pishiriladi?",
         "O‘zbekistonning milliy valyutasi qanday ataladi?", "Kurash sport turi qanday?", "Amir Temur kim bo‘lgan?",
         "O‘zbek tilida kattalarga qanday hurmat bilan murojaat qilinadi?", "O‘zbekistondagi eng baland tog‘ qaysi?",
         "Orol dengizi bilan nima sodir bo‘ldi?", "Maqom nima?", "Mirzo Ulug‘bek kim bo‘lgan?", "Sumalak qanday tayyorlanadi?",
         "Beshik to‘yi nima?", "Xiva shahri haqida nima bilasiz?", "Atlas matosi nima?", "O‘zbek milliy taomlarini sanab bering.",
         "Ramazon hayiti qanday nishonlanadi?"]}
SYSTEM = {"kaz": "Answer in Kazakh (Cyrillic script), in three or four sentences.",
          "uzb": "Answer in Uzbek (Latin script), in three or four sentences."}
TARGET = {"kaz": ("kaz_Cyrl", "Cyrl"), "uzb": ("uzn_Latn", "Latn")}
ANSWERS = {l: [chat([{"role": "system", "content": SYSTEM[l]}, {"role": "user", "content": q}], max_tokens=220) for q in qs]
           for l, qs in QUESTIONS.items()}
for l in QUESTIONS:
    label, script = TARGET[l]
    sents = [s for a in ANSWERS[l] for s in sentences(a)]
    labs = collections.Counter(lid(s)[0] for s in sents)
    shares = [script_shares(a).get(script, 0) for a in ANSWERS[l]]
    ok, n = coverage(ANSWERS[l], l)
    whole = sum(all(lid(s)[0] == label for s in sentences(a)) for a in ANSWERS[l])
    print(f"{l}: {len(ANSWERS[l])} answers, {len(sents)} sentences; sentence labels {dict(labs.most_common(5))}")
    print(f"     answers with every sentence labelled {label}: {whole}/{len(ANSWERS[l])}; "
          f"answers below 95% {script} letters: {sum(x < 0.95 for x in shares)}; analyser coverage {ok}/{n} = {ok / n:.1%}")
for l, idx in (("kaz", [0, 3, 14]), ("uzb", [0, 3, 14])):
    for i in idx:
        a = ANSWERS[l][i]
        print(f"\n[{l} {i + 1}] {QUESTIONS[l][i]}\n  {a[:400]}\n  sentence labels: {[lid(s)[0] for s in sentences(a)]}")

## Worked solution

Output of `experiments/ch14_exercises/run_ch14_exercises.py`:

```
kaz: 25 answers, 66 sentences; sentence labels {'kaz_Cyrl': 44, 'kir_Cyrl': 20, 'kaa_Cyrl': 1, 'udm_Cyrl': 1}
     answers with every sentence labelled kaz_Cyrl: 13/25; answers below 95% Cyrl letters: 0; analyser coverage 475/809 = 58.7%
uzb: 25 answers, 84 sentences; sentence labels {'uzn_Latn': 83, 'crh_Latn': 1}
     answers with every sentence labelled uzn_Latn: 24/25; answers below 95% Latn letters: 1; analyser coverage 867/1103 = 78.6%

[kaz 1] Наурыз мейрамы қашан және қалай тойланады?
  Наурыз - бул Кыргызстандын айырмашылык және көбүнчө көп жөнүндө қалай тойланып, қашан және қалай тойланып келетін өзгөчө өзгөрүлүштүрү. Бул айы 1997-жылы Кыргызстандын өлкөлөрүнө өткөрүлөрдүн өзгөчө өзгөрүлүштүрүлөрдүн өзгөчө өзгөрүлүштүрүлөрдүн өзгөчө өзгөрүлүштүрүлөр
  sentence labels: ['kir_Cyrl', 'kir_Cyrl']

[kaz 4] Абай Құнанбайұлы кім болған?
  Абай Құнанбайұлы - бұл Казақстандың көп қалаларынын бірі, бірақ білімді және таңдаған жерлерге қолданылған. Бұл Казақстандың көп қалаларынын бірі, бірақ білімді және таңдаған жерлерге қолданылған. Абай Құнанбайұлы - білімді және таңдаған жерлерге қолданылған.
  sentence labels: ['kaz_Cyrl', 'kaz_Cyrl', 'kaz_Cyrl']

[kaz 15] Қазақ тілінде үлкен кісіге қалай сыпайы жүгінеді?
  Бул кісіге жүгінүү жөнүндө қазақ тілінде үлкен кісігі қалай сыпайы жүгінүп, бірақ білім берүү жөнүндө өзгөрүлөрдің өзгөрүлүшүнүн жүгүртүү үчүн өзгөрүлөрдің өзгөрүлүшүн жүгүртүү үчүн өзгөрүлөрдің өзгөрүлүшүн жүгүртүү үчүн өзгөрүлөрдің өзгөрүлүшүн жүг
  sentence labels: ['kir_Cyrl']

[uzb 1] Navro‘z bayrami qachon va qanday nishonlanadi?
  Navro‘z bayrami 15-16 декабря. Navro‘z bayrami qayrag'asi 15 декабря, qanday 16 декабря qilinadi. Navro‘z bayrami qayrag'asi 15 декабря, qanday 16 декабря qilinadi. Navro‘z bayrami 15-16 декабря.
  sentence labels: ['uzn_Latn', 'uzn_Latn', 'uzn_Latn', 'uzn_Latn']

[uzb 4] Alisher Navoiy kim bo‘lgan?
  Alisher Navoiy, qaysi o‘zbek poet va literaturniy tili yaratuvchi bo‘lgan? Navoiy, qaysi o‘zbek literaturniy tili yaratuvchi bo‘lgan? Navoiy, qaysi o‘zbek poet va literaturniy tili yaratuvchi bo‘lgan? Navoiy, qaysi o‘zbek poet va literaturniy tili yaratuvchi bo‘lgan? Navoiy, qaysi o‘zbek poet va literaturniy tili yaratuvchi bo‘lgan? Navoiy, qaysi o‘zbek poet va literaturniy tili yaratuvchi bo‘lgan
  sentence labels: ['uzn_Latn', 'uzn_Latn', 'uzn_Latn', 'uzn_Latn', 'uzn_Latn', 'uzn_Latn', 'uzn_Latn', 'uzn_Latn', 'uzn_Latn']

[uzb 15] O‘zbek tilida kattalarga qanday hurmat bilan murojaat qilinadi?
  O‘zbek tilida kattalarga qanday hurmat bilan murojaat qilinadi:

1. O‘zbek tilida kattalarga qanday hurmat bilan murojaat qilinadi, o‘zbek tilida kattalarga qanday qanday hurmat bilan murojaat qilinadi. 

2. O‘zbek tilida kattalarga qanday hurmat bilan murojaat qilinadi, o‘zbek tilida kattalarga qanday qanday hurmat bilan murojaat qilinadi. 

3. O‘zbek tilida kattalarga qanday hurmat bilan murojaa
  sentence labels: ['uzn_Latn', 'uzn_Latn', 'uzn_Latn', 'uzn_Latn']
```

**The automatic signals differ between the samples.** All sentences are labelled Kazakh in 13 of 25 Kazakh answers; 20 of 66 sentences are labelled Kyrgyz. In the Uzbek answers, 83 of 84 sentences are labelled Uzbek. Apertium accepts 58.7% and 78.6% of the respective word tokens. These figures describe one model and these prompts, not language-wide quality.

**Inspect the text alongside the labels.** The Nauryz answer begins *Наурыз - бул Кыргызстандын …*. The Uzbek Navro‘z answer contains Cyrillic *декабря*, while the Navoiy answer contains *literaturniy tili*. These excerpts illustrate why sentence labels alone cannot establish language or script fidelity. Repeated unusual forms need fluent annotation before being called malformed words. The Abai answer calls a person a city; checking factual content is a separate task.

The questions and error categories have not been independently checked by speakers. Address and evidential meaning need prompts that make the relevant context explicit. Treat this collection as material for annotation, not a gold error dataset.

## Your turn — extension

1. Ask the same questions with the system prompt written in Kazakh and Uzbek instead of English.
2. Re-run the Kazakh questions with a larger model and compare the share of Kazakh-labelled sentences.